In [ ]:
import os
import shutil
import flopy
import pyemu
import numpy as np
import pandas as pd

import herebedragons as hbd
import matplotlib.pyplot as plt

from pathlib import Path

# Getting Started

In [ ]:
# folder containing original model files
org_d = Path('model/reactive')

# a dir to hold a copy of the org model files
tmp_d = Path('tmp')

if Path.exists(tmp_d):
    shutil.rmtree(tmp_d)
shutil.copytree(org_d,tmp_d)

# get executables if 00_model not run
hbd.prep_bins(tmp_d)

In [ ]:
# if you haven't run the first tutorial
# run the model once to make sure it works
# pyemu.os_utils.run("mf6rtm",cwd=tmp_d)

In [ ]:
# load simulation
sim = flopy.mf6.MFSimulation.load(sim_ws=tmp_d)
# load flow model
gwf = sim.get_model()

This model was built using a structured `DIS` grid. `PstFrom` is going to require the `flopy` model grid object to setup pilot points and spatially varying covariance:

In [ ]:
sr = gwf.modelgrid
sr

# Initializing PST FROM

In [ ]:
# specify a template directory (i.e. the PstFrom working folder)
template_ws = Path("pst_template")

# instantiate PstFrom
pf = pyemu.utils.PstFrom(original_d=tmp_d,       # where the model is stored
                            new_d=template_ws,   # the PEST template folder
                            remove_existing=True, # ensures a clean start
                            longnames=True,      # set False if using PEST/PEST_HP
                            spatial_reference=sr, #the spatial reference we generated earlier
                            zero_based=False, # does the MODEL use zero based indices? For example, MODFLOW does NOT
                            echo=False) # to stop PstFrom from writing lots of information to the notebook; experiment by setting it as True to see the difference; useful for troubleshooting

# Add Parameters

If you are familiar with highly parameterized groundwater models, you will know that even when considering only flow properties—such as hydraulic conductivity and storage or storativity—the number of parameters can already be large. When transport and reactive transport are included, the number of properties that can be parameterized can grow rapidly.

For the sake of this tutorial, we keep things relatively simple: we parameterize a selection of flow and boundary properties (hydraulic conductivity, storage, recharge, the GHB, and the well rates), a couple of transport properties (porosity and dispersivity), and one reaction property (the calcite reserve).

# Flow paramters
## Hydraulic conductivity

No one specifgied the `idomain` in the original model setup, so lets just create a "zone array". `PstFrom` expects this when we setup pilot points and so on later. Note that the shape of `ib` is the same as the shape of the model arrays:

In [ ]:
ib =gwf.dis.idomain.get_data()[0]
plt.imshow(ib)
cb = plt.colorbar()

In [ ]:
# we need to get the Hk files. For that we are gonna use a helper function that gets the filenames from a tag

tag = "npf_k"
files = hbd.get_input_filenames(tag, template_ws=template_ws, extension='.txt')

# flopy does not write nice and tidy array files so we are gonna do that here

# for f in files:
#     fpath = template_ws / f
#     hbd.tidy_array(fpath)


In [ ]:
files

In [ ]:
#Let's check one file against idomain
k = np.loadtxt(template_ws / files[0])

k.shape, ib.shape # all good!

In [ ]:
# spherical variogram for spatially varying parameters
v_pp = pyemu.geostats.SphVario(contribution=1.0, #sill
                                    a=500, # range of correlation; length units of the model. In our case 'meters'
                                    anisotropy=1, #name says it all
                                    bearing=0 #angle in degrees East of North corresponding to anisotropy ellipse
                                    )

# geostatistical structure for spatially varying parameters
pp_gs = pyemu.geostats.GeoStruct(variograms=v_pp, transform='log') 

pp_gs.plot()

In [ ]:
lb = 0.1
ub = 10.0
for f in files:
    tag = f.split('.')[1].replace("_",".")
    df_pp = pf.add_parameters(f,
                    zone_array=ib,
                    par_type="pilotpoints",
                    par_style="m",
                    geostruct=pp_gs,
                    par_name_base=tag,
                    pargp=tag,
                    lower_bound=lb,
                    upper_bound=ub,
                    ult_ubound = 100,
                    ult_lbound = 1e-2, # tiny
                    pp_options={"prep_hyperpars":False,
                                "pp_space":10} #in this case is cells
                    )
    df_cn = pf.add_parameters(f,
                    zone_array=ib,
                    par_type="constant",
                    par_style="m",
                    geostruct=pp_gs,
                    par_name_base=tag,
                    pargp=tag,
                    lower_bound=0.5,
                    upper_bound=2.0,
                    ult_ubound = 100,
                    ult_lbound = 1e-2, # tiny
                    )

In [ ]:
df_pp

In [ ]:
fig, ax = plt.subplots(1,1)
mv = flopy.plot.PlotMapView(gwf)
mv.plot_grid()
ax.scatter(df_pp.x, df_pp.y, s=10)

In [ ]:
df = pf.add_observations(f,prefix="hk",zone_array=ib)
df

In [ ]:
[f for f in os.listdir(template_ws) if f.endswith(".tpl")]

See those nice observation names with the "i" and "j" values baked in?

Now, back to parameterization...We are going to be repeating this multiplier-parameter scheme for each parameter type, so let's write a function.


In [ ]:
def add_mult_pars(f, lb=0.2, ub=5.0, ulb=0.01, uub=100, add_coarse=True):
    if isinstance(f,str):
        base = f.split(".")[1].replace("_","")
    else:
        base = f[0].split(".")[1]

    # pilot point (medium) scale parameters
    pf.add_parameters(f,
                        zone_array=ib,
                        par_type="pilotpoints",
                        geostruct=pp_gs,
                        par_name_base=base+"pp",
                        pargp=base+"pp",
                        lower_bound=lb, upper_bound=ub,
                        ult_ubound=uub, ult_lbound=ulb,
                        pp_options={"pp_space":10}) # `PstFrom` will generate a uniform grid of pilot points spaced `pp_space` cells apart
    if add_coarse==True:
        # constant (coarse) scale parameters
        pf.add_parameters(f,
                            zone_array=ib,
                            par_type="constant",
                            par_name_base=base+"cn",
                            pargp=base+"cn",
                            lower_bound=lb, upper_bound=ub,
                            ult_ubound=uub, ult_lbound=ulb)
    return

In [ ]:
# for Ss
tag = "sto_ss"
files = [f for f in os.listdir(template_ws) if tag in f.lower() and f.endswith(".txt")]

add_mult_pars(files[0], lb=0.5, ub=2.0, ulb=1e-7, uub=1e-1)

# For Sy
tag = "sto_sy"
files = [f for f in os.listdir(template_ws) if tag in f.lower() and f.endswith(".txt")]
add_mult_pars(files[0], lb=0.5, ub=2.0, ulb=0.001, uub=0.4)

In [ ]:
[f for f in os.listdir(template_ws) if f.endswith(".tpl")]

In [ ]:
# for Recharge; 
tag = "rcha_recharge"

files = [f for f in os.listdir(template_ws) if tag in f.lower() and f.endswith(".txt")]
files

In [ ]:
f = files[0]
add_mult_pars(f, lb=0.75, ub=1.25, ulb=0, uub=1e-3, add_coarse=False)

## List Files

Adding parameters from list-type files follows similar principles. As with observation files, they must be tabular. Certain columns are specified as index columns and are used to populate parameter names, as well as provide the parameters' spatial location. Other columns are specified as containing parameter values. 

Parameters can be `grid` or `constant`. As before, values can be assigned `directly`, as `multipliers` or as `additives`.

We will demonstrate for the boundary-condition input files. 

Starting off with GHBs. Let's inspect the folder. As you can see, there is a single input file (here we assume GHB parameters do not vary over time).

### GHB - aquifer boundary

In [ ]:
tag = "ghb_stress_period_data"
files = [f for f in os.listdir(template_ws) if tag in f.lower() and f.endswith(".txt")]
print(files)

Since these boundaries are likely to be very influential, we want to include a robust representation of their uncertainty - both head and conductance.  

Let's parameterize both GHB conductance and head:

 - For conductance, we use a `constant`-scale `multiplier` parameter.

 - For heads, multipliers are not ideal. Instead we use a `constant`-scale `additive` parameter.

 **ATTENTION!** 
 
 Additive parameters by default get assigned an initial parameter value of zero. This can be problematic later on when computing the derivatives. Be sure to either apply a parameter offset or use "absolute" increment types in the parameter group section when taking finite-difference derivatives of them. We use the absolute-increment approach for the well-rate decision variables in the optimization notebook (`06-opt`).


In [ ]:
tag = "ghb_stress_period_data"
files = [f for f in os.listdir(template_ws) if tag in f.lower() and f.endswith(".txt")]

for f in files:
    # constant-scale multiplier conductance parameter
    name = 'ghbcond'
    pf.add_parameters(f,
                        par_type="constant",
                        par_name_base=name+"cn",
                        pargp=name+"cn",
                        index_cols=[0,1,2],
                        use_cols=[4],  
                        lower_bound=0.1,upper_bound=10.0,
                        ult_lbound=0.01, ult_ubound=100) #absolute limits

    # constant-scale additive head parameter
    name = 'ghbhead'
    pf.add_parameters(f,
                        par_type="constant",
                        par_name_base=name+"cn",
                        pargp=name+"cn",
                        index_cols=[0,1,2],
                        use_cols=[3],
                        lower_bound=-2.0,upper_bound=2.0, 
                        par_style="a", 
                        transform="none",
                        ult_lbound=90, ult_ubound=100) 

### MAR Wells

In [ ]:
files = [f for f in os.listdir(template_ws) if "mar.wel_stress_period_data" in f and f.endswith(".txt")]
files.sort()
files

In [ ]:
f = files[0]
# add the parameters 
pf.add_parameters(filenames=f,
                    index_cols=[0,1,2], #reach number
                    use_cols=[3],   #columns with parameter values
                    par_type="grid",    
                    par_name_base="marwelgr",
                    pargp="marwelgr", 
                    transform="none",
                    par_style="direct",
                    upper_bound = 1200, lower_bound=0)

# f = files[-1]
# # add the parameters 
# pf.add_parameters(filenames=f,
#                     index_cols=[0,1,2], #reach number
#                     use_cols=[3],   #columns with parameter values
#                     par_type="grid",    
#                     par_name_base="marwelgr-rec",
#                     pargp="marwelgr-rec", 
#                     transform="none",
#                     par_style="direct",
#                     upper_bound = 1200, lower_bound=0)

### Dewatering wells

In [ ]:
files = [f for f in os.listdir(template_ws) if "dewater.wel_stress_period_data" in f and f.endswith(".txt")]
files.sort()
print(files)

f = files[0]

# add the parameters
_ = pf.add_parameters(filenames=f,
                    index_cols=[0,1,2], #reach number
                    use_cols=[3],   #columns with parameter values
                    par_type="grid",    
                    par_name_base="dewaterwelgr",
                    pargp="dewatwelgr", 
                    transform="none",
                    par_style="direct",
                    upper_bound = 0, lower_bound=-1600)

# f = files[-1]
# # add the parameters
# _ = pf.add_parameters(filenames=f,
#                     index_cols=[0,1,2], #reach number
#                     use_cols=[3],   #columns with parameter values
#                     par_type="grid",    
#                     par_name_base="dewaterwelgr-rec",
#                     pargp="dewatwelgr-rec", 
#                     transform="none",
#                     par_style="direct",
#                     upper_bound = 0, lower_bound=-1600)

# Reactive transport parameters

## Porosity

Let’s try working through one of the transport properties now. As you can see by inspecting the files, properties such as porosity (`mst_porosity`) or longitudinal dispersivity (`dsp_alh`) are defined separately for each transported species. This per-species duplication comes from mf6rtm building one GWT model per chemical component (each a clone of the tracer GWT). MODFLOW 6 lets each transport model carry its own values — genuinely useful for species-specific dispersion or diffusion — but porosity is a property of the medium, so physically we want it identical across species.

For most practical problems, however, it usually makes sense to use the same values for all species. Therefore, we have a couple of options for handling these properties—let’s focus on porosity for now:

1. Parameterize porosity for all species and define relationships in PEST (e.g., using tied parameters).
2. Parameterize porosity for only one species and then apply those parameters to the remaining species.

Here, we will choose the second option (easier :)). To do this, we will need to write a small function to “copy” the parameters before each forward run. Don’t worry—this is not particularly difficult to implement, and PyEMU is especially well suited for this task.


In [ ]:
# let's get a list of species first.

species = sim.model_names[1:] # the flow model is always fist so we can skip that one

species

We are gonna parameterize H and then use those parameters for the rest

In [ ]:
tag = "h.mst_porosity"
files = hbd.get_input_filenames(tag, template_ws=template_ws, extension='.txt')

# flopy does not write nice and tidy array files so we are gonna do that here

# for f in files:
#     fpath = template_ws / f
#     hbd.tidy_array(fpath)


lb = 0.25
ub = 1.25
for f in files:
    tag = f.split('.')[1].replace("_",".")
    df_pp = pf.add_parameters(f,
                    zone_array=ib,
                    par_type="pilotpoints",
                    par_style="m",
                    geostruct=pp_gs,
                    par_name_base=tag,
                    pargp=tag,
                    lower_bound=lb,
                    upper_bound=ub,
                    ult_ubound = 1, # max porosity is 1 but we don't wanna go over 0.6
                    ult_lbound = 5e-2, # tiny just in case
                    pp_options={"prep_hyperpars":False,
                                "pp_space":10} #in this case is cells (DIS grid)
                    )
    df_cn = pf.add_parameters(f,
                    zone_array=ib,
                    par_type="constant",
                    par_style="m",
                    geostruct=pp_gs,
                    par_name_base=tag,
                    pargp=tag,
                    lower_bound=lb,
                    upper_bound=ub,
                    ult_ubound = 1, # max poro is 1
                    ult_lbound = 0.05, # tiny
                    )
    
df = pf.add_observations(f,prefix="poro",zone_array=ib)
df

## Dispersivity

In [ ]:
tag = "h.dsp_alh"
files = hbd.get_input_filenames(tag, template_ws=template_ws, extension='.txt')

lb = 0.75
ub = 2
for f in files:
    tag = f.split('.')[1].replace("_",".")
    df_pp = pf.add_parameters(f,
                    zone_array=ib,
                    par_type="pilotpoints",
                    par_style="m",
                    geostruct=pp_gs,
                    par_name_base=tag,
                    pargp=tag.split(".")[0],
                    lower_bound=lb,
                    upper_bound=ub,
                    # ult_ubound = 0.65, 
                    # ult_lbound = 5e-2, # tiny just in case
                    pp_options={"prep_hyperpars":False,
                                "pp_space":10} #in this case is cells (DIS grid)
                    )
    df_cn = pf.add_parameters(f,
                    zone_array=ib,
                    par_type="zone",
                    par_style="m",
                    geostruct=pp_gs,
                    par_name_base=tag,
                    pargp=tag.split(".")[0],
                    lower_bound=lb,
                    upper_bound=ub,
                    # ult_ubound = 0.65,
                    # ult_lbound = 0.05,
                    )
df = pf.add_observations(f,prefix="dsp",zone_array=ib)
df


## Equilibrium phases: Calcite

We are going to parameterize the initial mass of Calcite that is available to reach equilibrium. 

In [ ]:
# spherical variogram for spatially varying parameters
v_pp = pyemu.geostats.SphVario(contribution=1.0, #sill
                                    a=750, # range of correlation; length units of the model. In our case 'meters'
                                    anisotropy=2, #name says it all
                                    bearing=0 #angle in degrees East of North corresponding to anisotropy ellipse
                                    )

# geostatistical structure for spatially varying parameters
pp_gs = pyemu.geostats.GeoStruct(variograms=v_pp, transform='log') 

pp_gs.plot()

In [ ]:
tag = "equilibrium_phases.calcite.m0."

files = hbd.get_input_filenames(tag, template_ws=template_ws, extension='.txt')
f=files[0]
arr = np.loadtxt(os.path.join(template_ws,f))
arr = arr.reshape(sr.nrow, sr.ncol)
np.savetxt(os.path.join(template_ws,f), arr, fmt='%1.6e')
# mf6rtm writes these files as tidy array already.. Winning

lb = 0.01
ub = 100

for f in files:
    tag = f.split('.txt')[0].replace("_",".").lower()
    df_pp = pf.add_parameters(f,
                    zone_array=ib,
                    par_type="pilotpoints",
                    geostruct=pp_gs,
                    par_name_base=tag,
                    pargp=tag,
                    lower_bound=lb,
                    upper_bound=ub,
                    # ult_ubound = 15, #
                    # ult_lbound = 1e-5, # tiny just in case
                    pp_options={"prep_hyperpars":True,
                                "pp_space":10} #in this case is cells (DIS grid)
                    )
    df_cn = pf.add_parameters(f,
                    zone_array=ib,
                    par_type="constant",
                    par_style="m",
                    geostruct=pp_gs,
                    par_name_base=tag,
                    pargp=tag,
                    lower_bound=lb,
                    upper_bound=ub,
                    # ult_ubound = 10, #
                    # ult_lbound = 1e-5, # tiny
                    )
df = pf.add_observations(f,prefix="calcite",zone_array=ib)
df

The `prep_hyperpars` flag did a number of things for us.  Primarily, it setup `pypestutils` input files that we can add additional parameters for.  Let's look at these:

In [ ]:
hyperpar_files = [f for f in os.listdir(pf.new_d) if tag in f]
hyperpar_files

To keep things managable in this example, we will parameterize the anisotropy of the pilot point variogram as a constant, that is a single value for all model cells so that we can vary the anisotropy.  We will also parameterize the bearing array with its own set of pilot points, which itself requires a variogram - lets define that here:

In [ ]:
bearing_v = pyemu.geostats.ExpVario(contribution=1,a=1000,anisotropy=3,bearing=90.0)
bearing_gs = pyemu.geostats.GeoStruct(variograms=bearing_v)

In [ ]:
# afile = tag+'.aniso.dat'
# atag = afile.split('.')[2].replace("_","-")+"-aniso"
# pf.add_parameters(afile,par_type="constant",par_name_base=atag,
#                   pargp=atag,lower_bound=-1.0,upper_bound=1.0,
#                   apply_order=1,
#                   par_style="a",transform="none",initial_value=0.0)
# pf.add_observations(afile, prefix=atag, obsgp=atag)

And now the bearing pilot points.  We will use the same pilot point spacing and will also use an additive parameter type for these parameters, with an uppper and lower bound of 45 and -45, respectively.  What this implies conceptually is that we can spatially change the bearing of the correlation ellispse between -45 and 45 degrees (since the original pilot point variogram has a bearing of 0.0)

In [ ]:
bfile = tag+'.bearing.dat'
btag = bfile.split('.')[2].replace("_","-")+"-bearing"
pf.add_parameters(bfile, par_type="pilotpoints", par_name_base=btag,
                  pargp=btag, lower_bound=-65,upper_bound=65,
                  par_style="a",transform="none",
                  pp_options={"pp_space":10,"try_use_ppu":True},
                  apply_order=1,geostruct=bearing_gs)
pf.add_observations(bfile, prefix=btag, obsgp=btag)     

bfile = tag+'.corrlen.dat'
btag = bfile.split('.')[2].replace("_","-")+"-corrlen"
pf.add_parameters(bfile, par_type="pilotpoints", par_name_base=btag,
                  pargp=btag, lower_bound=0.1,upper_bound=10,
                  par_style="m",transform="log",
                  pp_options={"pp_space":10,"try_use_ppu":True},
                  apply_order=1,geostruct=bearing_gs)
pf.add_observations(bfile, prefix=btag, obsgp=btag)     

In [ ]:
[f for f in os.listdir(template_ws) if f.endswith(".tpl")]

## Background water chemistry

The parameters so far leave the **background aquifer chemistry** fixed — yet our pre-development pH and well-chemistry measurements are, by construction, measurements *of* that background. If the background composition is certain, those observations carry no information: every realisation returns the same value, so there is nothing for history matching to fit.

So we let the major-ion background composition be uncertain. Each PHREEQC component has a uniform background initial-condition array (`<Comp>.ic_strt.txt`); a single `constant` multiplier per component turns its concentration into a parameter. Because `mf6rtm` re-equilibrates the chemistry through PHREEQC at every step, a different background alkalinity, sulfate, calcium or iron produces a different simulated pH and speciation — from day one. The pre-development chemistry observations then vary across the prior and can do real work in the history match.

We parameterise the acid–base and redox-relevant components — total carbon (C, alkalinity), sulfur (S), calcium (Ca) and iron (Fe) — each uncertain to about ±20%. Water, hydrogen, oxygen and the charge component are deliberately left out; PHREEQC re-balances charge on the scaled solutions.

In [ ]:
# Background water chemistry: make the major-ion composition uncertain.
# Each PHREEQC component has a uniform background IC array (<Comp>.ic_strt.txt);
# a constant multiplier turns its concentration into a parameter. mf6rtm
# re-equilibrates through PHREEQC every step, so the pre-development pH/chemistry
# observations gain prior spread (and data worth). H2O/H/O/Charge are left out --
# PHREEQC re-balances charge on the scaled solutions.
CHEM_COMPONENTS = ["C", "S", "Ca", "Fe"]   # alkalinity, sulfate, calcium, iron
LB, UB = 0.8, 1.25                          # ~+/-20% background uncertainty (log multiplier)

for comp in CHEM_COMPONENTS:
    pf.add_parameters(f"{comp}.ic_strt.txt",
                      par_type="constant",
                      par_style="m",
                      transform="log",
                      par_name_base=f"bg{comp.lower()}",
                      pargp=f"bg-{comp.lower()}",
                      lower_bound=LB, upper_bound=UB,
                      ult_lbound=1e-12)

# Pre- and post-processing functions

As we discussed earlier, we want to have some functions before and after we run the forward model. PyEMU is all about python, so we are going to do a forward_run.py script here using pyEMU. 

You will also certainly need to include some additional processing steps.  These are supported through the `PstFrom.pre_py_cmds` and `PstFrom.post_py_cmds`, which are lists for pre and post model run python commands and `PstFrom.pre_sys_cmds` and `PstFrom.post_sys_cmds`, which are lists for pre and post model run system commands (these are wrapped in `pyemu.os_utils.run()`).  

But what if your additional steps are actually an entire python function? Well, we got that too! `PstFrom.add_py_function()`. This method allows you to get functions from another (pre-prepared) python source file and add them to the `forward_run.py` script. We will demonstrate this to post-process model observations after each run.

Now let's see this py-sauce in action: we are going to add a little post-processing function to extract the final simulated water level for all model cells for the last stress period from the MF6 binary headsave file and save them to ASCII format so that PEST(++) can read them with instruction files.  And, while we are at it, let's also extract the global water budget info from the MF6 listing file and store it in dataframes - these are usually good numbers to watch!  We will need the simulated water level arrays later for sequential data assimilation (wouldn't it be nice if MF6 supported the writing of ASCII format head arrays?). 

All these function are stored in the "herebedragons.py" script which you can find in the tutorial folder.

In [ ]:
pf.extra_py_imports.append("flopy")
pf.extra_py_imports.append("shutil")
pf.extra_py_imports.append("from pathlib import Path")

pf.add_py_function("herebedragons.py","tidy_array()",is_pre_cmd=None)
pf.add_py_function("herebedragons.py","get_input_filenames()",is_pre_cmd=None)
pf.add_py_function("herebedragons.py","extract_layer_number()",is_pre_cmd=None)

pf.add_py_function("herebedragons.py","copy_parameterized_transport_files()",is_pre_cmd=True)
pf.mod_sys_cmds.append("mf6rtm")

In [ ]:
hbd.copy_parameterized_transport_files(ws=template_ws)

# The Forward Run Script

OK! So, we almost have all the base building blocks for a PEST(++) dataset. We have some (1) observations and some (2) parameters. We are still missing (3) the "forward run" script. Recall that in the PEST world, the "model" is not just the numerical model (e.g. MODFLOW). Instead it is a composite of the numerical model (or models) and pre- and post-processing steps, encapsulated in a "forward run" script which can be called from the command line. This command line instruction is what PEST(++) sees as "the model". During execution, PEST(++) writes values to parameter files, runs "the model", and then reads values from the observation files.

`PstFrom` automates the generation of such a script when constructing the PEST control file. The script is written to file named `forward_run.py`. It is written in Python (this is not a PEST(++) requirement, merely a convenience...we are working in Python after all...). 

How about we see that in action? Magic time! Let's create the PEST control file.

In [ ]:
pf.write_forward_run()

In [ ]:
pst = pf.build_pst()


In [ ]:
pst.write(template_ws / 'pest.pst',version=2)

In [ ]:
pst.parameter_data

# Add Observations

In [ ]:
# Add the post-processing functions that run after each forward model run and write the
# observation files PEST(++) reads: proc_ph_at_gde (minimum pH across the GDE drain),
# proc_chem_at_wells (well-cell water chemistry), and extract_hds_arrays_and_list_dfs (heads + budget).

pf.add_py_function("herebedragons.py","proc_ph_at_gde()",is_pre_cmd=False)
pf.add_py_function("herebedragons.py","proc_chem_at_wells()",is_pre_cmd=False)
pf.add_py_function("herebedragons.py","extract_hds_arrays_and_list_dfs()",is_pre_cmd=False)



In [ ]:
# run the func to create the file
fname, df = hbd.proc_ph_at_gde(wd=template_ws)

# add the observations
prefix = 'gde-ph'
_ = pf.add_observations(fname,
                        index_cols=['time'], # this column denote unique observations
                        use_cols='ph_min', #this are the actual values to track
                        prefix=prefix,
                        obsgp=prefix
                        )
files = [f for f in os.listdir(template_ws) if f.startswith("gde_ph")]



In [ ]:
prefix = 'min-ph'
_ = pf.add_observations("_min_ph.txt",
                        prefix=prefix,
                        obsgp=prefix
                        )
_

In [ ]:
# run the func to create the file
hbd.proc_chem_at_wells(wd=template_ws)
files = [f for f in os.listdir(template_ws) if f.startswith("chemwell_")]

for f in files:
    df = pd.read_csv(os.path.join(template_ws,f),index_col=0)
    _df = pf.add_observations(f,index_cols=["time"],use_cols=list(df.columns.values),
                        obsgp=f.split('.csv')[0].replace("_","-"),
                        prefix=f.split('.')[0].replace("_", "-"))
_df.head()

In [ ]:
hbd.test_extract_hds_arrays(template_ws)

In [ ]:
files = [f for f in os.listdir(template_ws) if f.startswith("hdslay")]
files

In [ ]:
for f in files:
    pf.add_observations(f,prefix=f.split(".")[0],obsgp=f.split(".")[0])

In [ ]:
for f in ["inc.csv","cum.csv"]:
    df = pd.read_csv(os.path.join(template_ws,f),index_col=0)
    pf.add_observations(f,index_cols=["totim"],use_cols=list(df.columns.values),
                        prefix=f.split('.')[0],obsgp=f.split(".")[0])

In [ ]:
df = pd.read_csv(os.path.join(template_ws,"gwf.obs.head.pit.csv"),index_col=0)
df.head()

In [ ]:
hds_df = pf.add_observations("gwf.obs.head.pit.csv", # the model output file to read
                            insfile="heads.csv.ins", #optional, the instruction file name
                            index_cols="time", #column header to use as index; can also use column number (zero-based) instead of the header name
                            use_cols=list(df.columns.values), #names of columns that include observation values; can also use column number (zero-based) instead of the header name
                            prefix="hdspit") #prefix to all observation names; choose something logical and easy to find. We use it later on to select observations

In [ ]:
hds_df.head()

In [ ]:
[f for f in os.listdir(template_ws) if f.endswith(".ins")]

In [ ]:
df = pd.read_csv(os.path.join(template_ws, "gwf.obs.drn.csv"), index_col=0)
df.head()

In [ ]:
# add the observations to pf
drn_df = pf.add_observations("gwf.obs.drn.csv", # the model output file to read
                            insfile="drn.csv.ins", #optional, the instruction file name
                            index_cols="time", #column header to use as index; can also use column number (zero-based) instead of the header name
                            use_cols=list(df.columns.values), #names of columns that include observation values; can also use column number (zero-based) instead of the header name
                            prefix="drn") #prefix to all observation names

drn_df

In [ ]:
# let's do some organization in the pst observation data for better filtering

obs = pst.observation_data
# obs['oname'] = obs['variable'].copy() #tidy up

pst = pf.build_pst()

pst.write(template_ws/'pest.pst',version=2)

# After Building the Control File

At this point, we can do some additional modifications that would typically be done that are problem specific.  Here we can tweak the setup, specifying things such as observation weights, parameter bounds, transforms, control data, etc.

Note that any modifications made after calling `PstFrom.build_pst()` will only exist in memory - you need to call `pf.pst.write()` to record these changes to the control file on disk.  Also note that if you call `PstFrom.build_pst()` after making some changes, these changes will be lost.  

For the current case, the main thing we haven't addressed are the forecasts. Observation weights are set later, in the prior-Monte-Carlo notebook.

We will do so now.

# Forecasts

For most models there is a forecast/prediction that someone needs. Rather than waiting until the end of the project, the forecast should be entered into your thinking and workflow __right at the beginning__.  Here we do this explicitly by monitoring the forecasts as "observations" in the control file.  This way, for every PEST(++) analysis we do, we can watch what is happening to the forecasts - #winning

The optional PEST++ `++forecasts` control variable allows us to provide the names of one or more observations featured in the "observation data" section of the PEST control file; these are treated as predictions in FOSM predictive uncertainty analysis by PESTPP-GLM. It is also a convenient way to keep track of "forecast" observations (makes post-processing a wee bit easier later on).

In [ ]:
obs = pst.observation_data

obs.usecol.unique()
obs

obs[obs.usecol == 'drn-gde']

In [ ]:
obs.oname.unique()

In [ ]:
dobs = obs.loc[obs.usecol=="drn-gde",:].copy()
dobs['time'] = dobs.time.astype(int)
dobs.sort_values(by='time',inplace=True)
# dobs
dobs.obsnme.iloc[1], dobs.obsnme.iloc[-1]

In [ ]:
forecasts = [dobs.obsnme.iloc[1], dobs.obsnme.iloc[-1]]
forecasts

In [ ]:
hdspitobs = obs[(obs.oname == 'hdspit') & (obs.time == '3651') & (obs.i=="49") & (obs.j=="49")].obsnme.values
hdspitobs
# obs[(obs.oname == 'hdspit')]

In [ ]:
forecasts.append(hdspitobs[0])
forecasts

In [ ]:
phobs= obs.loc[(obs.oname == 'gde-ph'),:].copy()
phobs['time'] = phobs.time.astype(int)
phobs.sort_values(by='time',inplace=True)
# phobs.obsnme.iloc[1], phobs.obsnme.iloc[-1]
phobs

In [ ]:
forecasts.extend(phobs.obsnme.values[1:])
forecasts

In [ ]:
# forecasts.append(obs.loc[(obs.oname == 'min-ph'),'obsnme'].values[0])

In [ ]:
fobs = obs.loc[forecasts,:]
fobs

In [ ]:
pst.pestpp_options['forecasts'] = forecasts

In [ ]:
pst.pestpp_options["overdue_giveup_fac"] = 10
pst.pestpp_options["overdue_giveup_minutes"] = 100

In [ ]:
pst.write(os.path.join(template_ws, 'pest.pst'),version=2)

# Draw Prior Ensemble

In [ ]:
# build the prior covariance matrix and store it as a compressed binary file (otherwise it can get huge!)
# depending on your machine, this may take a while...
if pf.pst.npar < 35000:  #if you have more than about 35K pars, the cov matrix becomes hard to handle
    cov = pf.build_prior(fmt='coo', filename=Path(template_ws,"prior_cov.jcb"))
    # and take a peek at a slice of the matrix
    try:
        x = cov.x.copy()
        x[x==0] = np.NaN
    except:
        pass
    pf.pst.pestpp_options["parcov"] = "prior_cov.jcb"

In [ ]:
plt.imshow(x[:10,:10])

In [ ]:
pe = pf.draw(num_reals=1000, use_specsim=False) # draw parameters from the prior distribution
pe.enforce() # enforces parameter bounds


pe.to_binary(Path(template_ws,"prior_pe.jcb")) #writes the parameter ensemble to binary file


pst.pestpp_options["ies_par_en"] = "prior_pe.jcb" #tells pestpp to use prior_pe
pst.pestpp_options["ies_num_reals"] = 100

pst.write(Path(template_ws,"pest.pst"),version=2)

# Test runs

It is good practice to do some test runs and sanity checks. This part is especially good for checking parameter ranges and adjust your conceptual and /or numerical understanding of your site. 



In [ ]:
pyemu.os_utils.run('pestpp-ies pest.pst', cwd=template_ws)

In [ ]:
pst = pyemu.Pst(str(template_ws/"pest.pst"))
pst.phi < 1e-15 # sanity check

# Check Fields

In [ ]:
pst.control_data.noptmax = -2
pst.pestpp_options["ies_run_realname"] = 2

pst.write(template_ws / "test.pst",version=2)


In [ ]:
pyemu.os_utils.run('pestpp-ies test.pst', cwd=template_ws)

In [ ]:
sim = flopy.mf6.MFSimulation.load(sim_ws = template_ws,
                                sim_name = 'gwf', 
                                version='mf6',
                                exe_name='mf6',
                                verbosity_level=0)
gwf = sim.get_model("gwf")
gwt = sim.get_model("h")

poro = gwt.mst.porosity.get_data()
k = gwf.npf.k.get_data()

# Calcite initial moles is a mf6rtm external array (not in the flopy sim)
nrow, ncol = gwf.dis.nrow.get_data(), gwf.dis.ncol.get_data()
calcite = np.log(np.loadtxt(template_ws / 'equilibrium_phases.Calcite.m0.layer1.txt').reshape(nrow, ncol))

plot_arrays = [k, poro, calcite]
plot_titles = ['H$_k$ (m/d)', 'Porosity (-)', 'Calcite (mol) - log']

fig, axs = plt.subplots(1, len(plot_arrays), figsize=(4 * len(plot_arrays), 3),
                        constrained_layout=True, squeeze=True, sharex=True, sharey=True)
for ax, a, t in zip(axs, plot_arrays, plot_titles):
    mv = flopy.plot.PlotMapView(gwf, layer=0, ax=ax)
    mv.plot_grid(alpha=0.5)
    arr = mv.plot_array(a,  cmap="RdYlBu_r")
    cbar = plt.colorbar(arr, ax=ax, shrink=1)
    ax.set_title(t)
    ax.set_xlabel("X (m)")
    ax.set_ylabel("Y (m)")
    ax.set_aspect('equal')
    ax.set_xticks([])
    ax.set_yticks([])